In [ ]:
import pandas as pd
import glob

# merge tracks and tracksMeta files with proper id and frame alignment
def merge_tracks_and_meta(tracks_files, tracks_meta_files):
    combined_tracks_data = pd.DataFrame()
    combined_meta_data = pd.DataFrame()

    id_offset = 0  # Start with ID as 0
    frame_offset = 0  # Start with frame as 0

    # Process each pair of track and meta files
    for file_index, (track_file, meta_file) in enumerate(zip(tracks_files, tracks_meta_files), start=1):
        print(f"Processing: {track_file} and {meta_file}")

        # Load track and meta data
        tracks_data = pd.read_csv(track_file)
        meta_data = pd.read_csv(meta_file)

        # Adjust IDs in both files
        tracks_data["id"] += id_offset
        meta_data["id"] += id_offset

        # Adjust frame numbers in tracks data
        tracks_data["frame"] += frame_offset

        # Update IDs in related columns
        related_id_columns = [
            "precedingId", "followingId", "leftPrecedingId", "leftAlongsideId",
            "leftFollowingId", "rightPrecedingId", "rightAlongsideId", "rightFollowingId"
        ]

        for col in related_id_columns:
            if col in tracks_data.columns:
                tracks_data[col] = tracks_data[col].apply(lambda x: x + id_offset if x > 0 else x)

        # Update offsets for the next file
        id_offset = tracks_data["id"].max() + 1
        frame_offset = tracks_data["frame"].max() + 1

        # Combine the current data with the final output
        combined_tracks_data = pd.concat([combined_tracks_data, tracks_data], ignore_index=True)
        combined_meta_data = pd.concat([combined_meta_data, meta_data], ignore_index=True)

    return combined_tracks_data, combined_meta_data


# File patterns for track and meta files
tracks_file_pattern = "*_tracks.csv"
tracks_meta_file_pattern = "*_tracksMeta.csv"

# Get all the track and meta files
tracks_files = sorted(glob.glob(tracks_file_pattern))
tracks_meta_files = sorted(glob.glob(tracks_meta_file_pattern))

# Check if the number of track and meta files are the same
if len(tracks_files) != len(tracks_meta_files):
    raise ValueError("The number of track and meta files must match!")

# Merge the files
merged_tracks_data, merged_tracks_meta_data = merge_tracks_and_meta(tracks_files, tracks_meta_files)

# Save the final combined data to new files
merged_tracks_data.to_csv("merged_tracks.csv", index=False)
merged_tracks_meta_data.to_csv("merged_tracksMeta.csv", index=False)

print("Merging complete. Files saved as 'merged_tracks.csv' and 'merged_tracksMeta.csv'.")


Merging complete. Files saved as 'merged_tracks.csv' and 'merged_tracksMeta.csv'.
